# Encontro 5 — pandas: as cinco operações

**Programação Aplicada ao Direito** · LabDados / FGV Direito SP

A base de hoje tem **7.000 sentenças de primeiro grau**, de 24 tribunais estaduais, de uma
pesquisa real. Cada linha é um processo; cada coluna, uma informação objetiva extraída da sentença.

Hoje a gente vê **cinco operações** — são as cinco que resolvem quase tudo em qualquer base —
e, no fim, uma sexta para juntar duas tabelas:

| # | Verbo | O que faz | No pandas |
|---|---|---|---|
| 1 | **filter** | escolher **linhas** | `processos[...]` · `processos.query(...)` |
| 2 | **select** | escolher **colunas** | `processos[["col1", "col2"]]` |
| 3 | **mutate** | criar ou alterar coluna | `processos.assign(...)` |
| 4 | **arrange** | ordenar | `processos.sort_values(...)` |
| 5 | **summarise** | resumir em um número | `.mean()` · `.groupby(...)` |
| 6 | *join* | juntar duas tabelas | `processos.merge(...)` |

**Como preparar o Colab:** clique no ícone de **pasta**, na barra da esquerda, e arraste os
arquivos `processos.csv` e `dispositivos.csv` para dentro. Depois rode a célula abaixo.

Em cada exercício, **o código está no enunciado**. Olhe para ele, digite na célula e rode.
Digitar (em vez de copiar e colar).

In [ ]:
import pandas as pd

processos = pd.read_csv("processos.csv", dtype={"ganhou_autor": "boolean"})

processos.shape

> Se apareceu `(7000, 11)`, está tudo certo: 7.000 linhas e 11 colunas.
>
> Se apareceu um erro vermelho com `FileNotFoundError`, os arquivos não foram para o Colab —
> repita o passo da pasta.

### O que é `dtype={"ganhou_autor": "boolean"}`

| | |
|---|---|
| `dtype` | quer dizer **tipo** |
| o que essa parte faz | manda o pandas ler `ganhou_autor` como verdadeiro/falso |
| por que é preciso avisar | a coluna tem **três** valores: `True`, `False` e **vazio** (os acordos) |
| sem o aviso | o pandas lê a coluna como texto — e texto não tem média: a Seção 5 quebraria |
| por que `"boolean"` entre aspas | é o tipo do pandas, que aceita vazio; o `bool` do Python não aceita |

### Recap: os tipos de uma coluna

**O tipo é da coluna inteira, não de cada célula.** Uma coluna é de texto ou é de número.

| Tipo | Nome no pandas | Nesta base | Serve para |
|---|---|---|---|
| texto | `object` (ou `str`) | `tribunal`, `reu`, `resultado` | contar, comparar, procurar palavra |
| inteiro | `int64` | `ano`, `n_chars` | somar, ordenar, tirar média |
| decimal | `float64` | `valor_condenacao` | somar, ordenar, tirar média |
| verdadeiro/falso | `boolean` | `ganhou_autor` | contar, tirar **proporção** |
| data | `datetime64` | *(não há nesta base)* | extrair ano e mês, calcular prazo |

**Dois avisos**

- **Vazio ≠ zero ≠ falso.** Aparece como `NaN` (números) ou `<NA>` (resto), e **fica de fora das contas**.
- **Média de coluna verdadeiro/falso = proporção de `True`.** `0.579` lê-se 57,9%.

Para ver o tipo de cada coluna: `processos.dtypes` — é a célula **0.3**.

---
## Seção 0 — Conhecer a base

Antes de perguntar qualquer coisa a uma base, a gente olha para ela. **Estas células já estão
prontas: é só rodar** e ler a saída.

### 0.1 — O tamanho

`shape` é um **atributo**: vem sem parênteses, porque a informação já está guardada.

In [ ]:
processos.shape

### 0.2 — As primeiras linhas

`head()` é um **método**: vem com parênteses, porque ele monta as linhas na hora.

In [ ]:
processos.head(3)

### 0.3 — Os nomes e os tipos das colunas

In [ ]:
processos.dtypes

### 0.4 — O que está faltando

Célula vazia na base tem nome: **dado faltante**. Aqui a gente conta quantos há em cada coluna.

In [ ]:
processos.isna().sum()

### 0.5 — Como se distribui uma coluna

`value_counts()` conta quantas vezes cada valor aparece.

In [ ]:
processos["resultado"].value_counts()

> Repare em dois números que voltam o dia inteiro:
>
> - **2.697 acordos homologados**. Em acordo não há vencedor: por isso a coluna `ganhou_autor`
>   fica **vazia** nesses processos, e não `False`.
> - **`valor_condenacao` está vazia** quando a sentença não mandou pagar nada — vazia, não zero.

---
## Seção 1 — filter: escolher linhas

> **filter** (filtrar) fica só com as linhas que satisfazem uma condição. A tabela sai com **menos linhas e as mesmas colunas**.

Duas formas de escrever a mesma coisa:

- **colchete com a condição:** `processos[processos["ano"] == 2022]`
- **`.query()`**, com a condição escrita entre aspas: `processos.query('ano == 2022')`

Dentro do `.query()` o nome da coluna vai solto, sem colchete e sem aspas. Texto vai entre
aspas duplas, número vai sem nada.

### Exercício 1.1 — Quantos processos são de 2022?

A base tem 7.000 linhas. Quantas são de 2022?

Digite na célula abaixo:

```
processos[processos["ano"] == 2022]
```

**Esperado:** no rodapé da tabela, `1615 rows × 11 columns`.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 1.2 — Quantos processos são do TJRJ?

Agora a mesma ideia com `.query()`, e com uma coluna de texto. Quantos processos são do TJRJ?

Digite na célula abaixo:

```
processos.query('tribunal == "TJRJ"')
```

**Esperado:** `1906 rows × 11 columns`.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 1.3 — Quantos processos do TJRJ são de 2022?

Do TJRJ **e** de 2022. As duas condições entram na mesma frase, ligadas por `and`.

Digite na célula abaixo:

```
processos.query('tribunal == "TJRJ" and ano == 2022')
```

Com colchete, a mesma coisa seria assim:

```
processos[(processos["tribunal"] == "TJRJ") & (processos["ano"] == 2022)]
```

Cada condição entre parênteses, e `&` no lugar do `and`. Devolve exatamente a mesma tabela — e é por isso que a gente prefere o `query` quando há mais de uma condição.

**Esperado:** `781 rows × 11 columns`.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 1.4 — Em quantas sentenças o autor ganhou alguma coisa?

Quando os valores aceitos são vários, `in` com uma lista evita escrever `or` três vezes.

Queremos as sentenças em que o autor ganhou alguma coisa — procedentes e parcialmente procedentes.

Digite na célula abaixo:

```
processos.query('resultado in ["PROCEDENTE", "PARCIALMENTE PROCEDENTE"]')
```

Com colchete, a mesma coisa seria assim:

```
processos[processos["resultado"].isin(["PROCEDENTE", "PARCIALMENTE PROCEDENTE"])]
```

`isin` é o `in` do `query` escrito como método da coluna: *está em alguma dessas opções?*

**Esperado:** `1270 rows × 11 columns` (693 procedentes + 577 parcialmente procedentes).

In [ ]:
#COMPLETAR ABAIXO

---
## Seção 2 — select: escolher colunas

> **select** (selecionar) fica só com as colunas pedidas. A tabela sai com **as mesmas linhas e menos colunas**.

Aqui mora a única pegadinha de sintaxe do dia, e é sobre o número de colchetes:

- **um colchete** → devolve uma **Series**: uma coluna só, sem cabeçalho de tabela.
- **dois colchetes** → devolve um **DataFrame**: uma tabela, mesmo que com uma coluna só.

O colchete de dentro é a **lista** de colunas que você quer.

### Exercício 2.1 — Quais são os tribunais, linha a linha?

Peça a coluna `tribunal`, sozinha.

Digite na célula abaixo:

```
processos["tribunal"]
```

**Esperado:** uma lista de 7.000 valores, terminando com `Name: tribunal, Length: 7000, dtype: object`. Isso é uma **Series**.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 2.2 — Como fica a tabela só com tribunal e ano?

Agora `tribunal` e `ano`, as duas. Repare no colchete duplo.

Digite na célula abaixo:

```
processos[["tribunal", "ano"]]
```

**Esperado:** `7000 rows × 2 columns` — agora é uma tabela.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 2.3 — Quantas linhas e colunas ficam na tabela enxuta?

Selecione quatro colunas e guarde o resultado no nome `enxuta`. A segunda linha mostra o tamanho.

Digite na célula abaixo:

```
enxuta = processos[["processo", "tribunal", "ano", "resultado"]]
enxuta.shape
```

**Esperado:** `(7000, 4)`.

In [ ]:
#COMPLETAR ABAIXO

#### Confira o que ficou em `enxuta`

A tabela `enxuta` continua existindo depois da célula anterior. Esta célula já está pronta: é só rodar.

In [ ]:
enxuta.head(2)

#### Antes de filtrar, veja os setores

Para filtrar por setor é preciso saber quais valores existem. Esta célula já está pronta: é só rodar.

In [ ]:
processos["setor"].value_counts(dropna=False)

### Exercício 2.4 — Em que tribunais estão as ações contra empresas aéreas, e com que resultado?

As operações se encadeiam: primeiro escolhe as linhas, depois escolhe as colunas.

Queremos, das ações contra empresas de transporte aéreo, só o tribunal e o resultado.

Digite na célula abaixo:

```
processos.query('setor == "Transporte aéreo"')[["tribunal", "resultado"]]
```

**Esperado:** `271 rows × 2 columns`.

In [ ]:
#COMPLETAR ABAIXO

---
## Seção 3 — mutate: criar coluna

> **mutate** (`assign`, no pandas) cria uma coluna nova a partir das que já existem. A tabela sai com **as mesmas linhas e uma coluna a mais**.

`assign` não altera a tabela original: ele devolve **uma cópia com a coluna nova**. Se você quiser
guardar, tem que dar um nome ao resultado.

A conta vale para a coluna inteira de uma vez — não existe `for` aqui.

### Exercício 3.1 — Há quantos anos cada processo foi ajuizado?

Crie a coluna `idade`, que é 2026 menos o ano. A parte final pede só duas colunas, para caber na tela.

Digite na célula abaixo:

```
processos.assign(idade = 2026 - processos["ano"])[["ano", "idade"]].head(3)
```

Essa linha tem quatro pedaços, e cada um recebe o resultado do anterior, da esquerda para a direita:

| Pedaço | O que acontece | O que sai |
|---|---|---|
| `processos` | a tabela como está | 7.000 × 11 |
| `.assign(idade = 2026 - processos["ano"])` | cria a coluna `idade`. O nome novo vai à esquerda do `=`, **sem aspas**; à direita, a conta, feita na coluna inteira de uma vez | 7.000 × 12 |
| `[["ano", "idade"]]` | escolhe duas colunas dessa tabela nova. Dois colchetes, porque o que queremos é uma tabela | 7.000 × 2 |
| `.head(3)` | mostra só as três primeiras linhas | 3 × 2 |

Duas coisas para reparar: `processos` **não mudou** — o `assign` devolveu uma cópia, e essa cópia existe só durante a linha; e a conta `2026 - processos["ano"]` não precisou de nenhum `for`, porque valeu para as 7.000 linhas de uma vez.

**Esperado:** três linhas: 2021 → 5, 2015 → 11, 2020 → 6.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 3.2 — Quais processos são de 2022 para cá?

Crie a coluna `recente`, verdadeira quando o ano é 2022 ou mais, e mostre as cinco primeiras linhas com o ano ao lado, para conferir.

Digite na célula abaixo:

```
processos.assign(recente = processos["ano"] >= 2022)[["ano", "recente"]].head(5)
```

**Esperado:** cinco linhas: 2021, 2015, 2020 e 2021 com `False`, e 2022 com `True`.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 3.3 — Quais sentenças terminaram em acordo homologado?

A comparação também funciona com texto. Crie a coluna `acordo`, verdadeira quando o resultado foi acordo homologado, e mostre as quatorze primeiras linhas ao lado do resultado.

Digite na célula abaixo:

```
processos.assign(acordo = processos["resultado"] == "ACORDO HOMOLOGADO")[["resultado", "acordo"]].head(14)
```

**Esperado:** quatorze linhas. Só a última, um acordo homologado, fica `True` — todas as outras, `False`.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 3.4 — Quais processos têm 5 anos ou mais?

Agora a segunda coluna depende da primeira: `antigo` só existe se `idade` já existir. A saída é guardar a tabela com a coluna nova num nome, e usar esse nome na linha seguinte.

Digite na célula abaixo:

```
com_idade = processos.assign(idade = 2026 - processos["ano"])
com_idade.assign(antigo = com_idade["idade"] >= 5)[["ano", "idade", "antigo"]].head(3)
```

Repare por que foram duas linhas: na primeira, `idade` passa a existir dentro de `com_idade`; na segunda, dá para usá-la. Se você tentasse fazer tudo de uma vez, o `processos` original ainda não teria a coluna `idade`.

**Esperado:** três linhas: 2021 com idade 5, 2015 com 11 e 2020 com 6 — as três `True`.

In [ ]:
#COMPLETAR ABAIXO

#### Quantos processos têm 5 anos ou mais?

A coluna `antigo` responde linha a linha. Para o total, filtre e conte os números de processo distintos. Esta célula já está pronta: é só rodar.

In [ ]:
com_idade.query("idade >= 5")["processo"].nunique()

---
## Seção 4 — arrange: ordenar

> **arrange** (`sort_values`) coloca as linhas em ordem. A tabela sai com **as mesmas linhas e as mesmas colunas** — só muda a ordem.

Por padrão a ordem é crescente. Para inverter, `ascending=False`.

Ordenar é o que responde a pergunta "qual é o maior?" sem precisar procurar com o olho.

### Exercício 4.1 — Quais são os processos mais antigos da base?

Ordene pela coluna `ano` e mostre as três primeiras linhas.

Digite na célula abaixo:

```
processos.sort_values("ano")[["processo", "ano"]].head(3)
```

**Esperado:** três processos de **2014**, o ano mais antigo da base.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 4.2 — Quais foram as maiores condenações?

Agora ao contrário: da maior condenação para a menor.

Digite na célula abaixo:

```
processos.sort_values("valor_condenacao", ascending=False)[["tribunal", "valor_condenacao"]].head(3)
```

**Esperado:** TJMA com 237.600, TJES com 170.000 e TJRS com 100.000.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 4.3 — Quais são as sentenças mais longas?

`n_chars` é o tamanho da sentença em caracteres. Quais são as três maiores?

Digite na célula abaixo:

```
processos.sort_values("n_chars", ascending=False)[["tribunal", "n_chars"]].head(3)
```

**Esperado:** TJMG com 586.142 caracteres, e dois do TJSC com 117.335 e 114.800. A primeira é quase cinco vezes a segunda — vale desconfiar desse caso.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 4.4 — Dentro de cada ano, quais foram as maiores condenações?

Primeiro pelo ano, do mais antigo para o mais novo; dentro de cada ano, da maior condenação para a menor.

Quando são duas colunas, os dois argumentos viram listas. Aqui não peça `head`: deixe a tabela inteira aparecer, que o pandas já mostra o começo e o fim.

Digite na célula abaixo:

```
processos.sort_values(["ano", "valor_condenacao"], ascending=[True, False])[["ano", "valor_condenacao"]]
```

Quando a tabela é grande, o pandas mostra sozinho as primeiras e as últimas linhas, com `...` no meio. As vazias caem no fim de cada ano, mesmo na ordem decrescente.

**Esperado:** as cinco primeiras linhas (2014, começando em 16.000), reticências, e as cinco últimas (2025, com o valor vazio). No rodapé, `7000 rows × 2 columns`.

In [ ]:
#COMPLETAR ABAIXO

---
## Seção 5 — summarise: resumir em um número

> **summarise** (resumir) transforma muitas linhas em **um número**: uma média, uma contagem, uma mediana. Com `groupby`, faz isso **dentro de cada grupo**.

Quem sabe tirar média é a **coluna**, não a tabela: por isso o `.mean()` vem depois do colchete.

`groupby` sozinho não mostra nada — ele só separa a base em montinhos. Quem devolve o número é a
operação que vem depois: `.mean()`, `.size()`, `.count()`.

### Exercício 5.1 — Qual é a condenação média?

Qual a condenação média, considerando só as sentenças que condenaram a pagar algo?

Digite na célula abaixo:

```
processos["valor_condenacao"].mean()
```

**Esperado:** `5451.77...` — R$ 5.451, em média. As linhas vazias ficam de fora da conta sozinhas.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 5.2 — Quais tribunais aparecem mais na base?

Quantos processos tem cada tribunal? `value_counts()` já vem ordenado do maior para o menor.

Digite na célula abaixo:

```
processos["tribunal"].value_counts().head(3)
```

**Esperado:** TJRJ 1906, TJMA 762, TJMG 627.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 5.3 — Contra que setores o autor mais ganha?

Aqui aparece o `groupby`. Em quais setores econômicos o autor mais ganha?

`ganhou_autor` é verdadeiro ou falso, e a média de uma coluna de verdadeiro/falso é a **proporção** de verdadeiros.

Digite na célula abaixo:

```
processos.groupby("setor")["ganhou_autor"].mean().sort_values(ascending=False).round(3)
```

**Esperado:** transporte aéreo no topo, com **0.579**, e bureau de crédito no fim, com **0.095**.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 5.4 — Sobre quantos casos essa taxa de êxito é calculada?

`agg` vem de *aggregate*, agregar. Serve para pedir **mais de um resumo de uma vez**: no lugar de `.mean()`, escreva `.agg([...])` e ponha dentro dos colchetes a **lista** dos resumos que você quer. Cada item da lista vira **uma coluna** do resultado; as linhas continuam sendo os grupos.

| Se você escreve | O que volta |
|---|---|
| `.mean()` | um número por grupo |
| `.agg(["mean"])` | o mesmo número, mas já numa tabela de uma coluna |
| `.agg(["size", "count", "mean"])` | uma tabela com três colunas, uma para cada resumo |

Os nomes que cabem dentro de `agg` são os resumos de sempre: `size`, `count`, `mean`, `median`, `min`, `max`, `sum`, `std`. Aqui interessam três: `size` conta as linhas do grupo; `count` conta só as linhas em que `ganhou_autor` **não está vazia**; `mean` é a proporção.

Digite na célula abaixo:

```
processos.groupby("setor")["ganhou_autor"].agg(["size", "count", "mean"]).round(3)
```

**Esperado:** em transporte aéreo, `size` = **271** e `count` = **195**. Os 76 que somem são os acordos, em que `ganhou_autor` está vazia. Por isso a frase certa é: *entre as ações contra aéreas **julgadas no mérito**, o autor ganha 58%*.

In [ ]:
#COMPLETAR ABAIXO

---
## Seção 6 — join: juntar duas tabelas

> **join** (`merge`) cola duas tabelas usando a coluna que identifica a linha nas duas — a **chave**. A tabela sai com **as mesmas linhas e as colunas das duas**.

O texto da decisão não está em `processos`: ele ficou numa segunda tabela, `dispositivos`, que tem
o número do processo, o autor, o réu e o **dispositivo** — o trecho final da sentença.

A chave é a coluna `processo`, o número CNJ: ela identifica a linha **sem ambiguidade nas duas
tabelas**. Juntar por uma coluna que se repete, como o tribunal, colaria processos que nada têm a
ver uns com os outros.

`how="left"` significa: *mantenha todas as linhas da tabela da esquerda*. É o que a gente quer
quase sempre — a base de processos não pode encolher.

### Exercício 6.1 — O que tem na tabela dos dispositivos?

A segunda tabela ainda não está aberta: no começo do notebook o `read_csv` leu só o arquivo `processos.csv`. Cada tabela precisa do seu próprio `read_csv` e do seu próprio nome.

Leia `dispositivos.csv` agora e olhe as duas primeiras linhas.

Digite na célula abaixo:

```
dispositivos = pd.read_csv("dispositivos.csv")

dispositivos.head(2)
```

**Esperado:** duas linhas e quatro colunas: `processo`, `autor_tipo`, `reu` e `dispositivo`.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 6.2 — Como trazer o dispositivo para dentro da base?

`on` é **a coluna pela qual as duas tabelas se encontram** — a chave. `on="processo"` manda o pandas, para cada linha de `processos`, procurar em `dispositivos` a linha que tem o **mesmo número de processo** e colar as duas lado a lado. O nome que vai dentro do `on` precisa existir nas duas tabelas e estar escrito igual nas duas.

Junte as duas tabelas pela chave. `autor_tipo` e `reu` já existem em `processos`, então traga da segunda tabela **só a chave e o que falta** — por isso o colchete duplo dentro do merge.

Digite na célula abaixo:

```
completa = processos.merge(dispositivos[["processo", "dispositivo"]], on="processo", how="left")
completa.shape
```

**Esperado:** `(7000, 12)`: as mesmas 7.000 linhas, e uma coluna a mais.

In [ ]:
#COMPLETAR ABAIXO

#### E se eu trouxesse a tabela inteira?

A tabela `dispositivos` também tem `autor_tipo` e `reu` — que já existem em `processos`. Se o merge for feito sem o colchete duplo, o pandas não escolhe entre as duas versões: ele **traz as duas** e marca a origem com um sufixo, `_x` para a tabela da esquerda e `_y` para a da direita. Veja o que sai: `autor_tipo_x` e `autor_tipo_y`, `reu_x` e `reu_y` — **14 colunas** no lugar de 12, e duas delas repetidas. É por isso que a gente pede só a chave e o que falta.

In [ ]:
processos.merge(dispositivos, on="processo", how="left").columns

### Exercício 6.3 — Algum processo ficou sem o dispositivo?

Depois de um `merge`, duas perguntas: **perdi linha?** (o `shape` acima já respondeu) e **ficou alguém sem o dado novo?**

Digite na célula abaixo:

```
completa["dispositivo"].isna().sum()
```

**Esperado:** `1` — de 7.000 processos, só um ficou sem o texto do dispositivo.

In [ ]:
#COMPLETAR ABAIXO

### Exercício 6.4 — O que decidiram as sentenças procedentes?

Agora dá para ler a decisão ao lado do resultado. Veja duas sentenças procedentes.

Digite na célula abaixo:

```
completa.query('resultado == "PROCEDENTE"')[["processo", "dispositivo"]].head(2)
```

**Esperado:** duas linhas, com o número do processo e o trecho final da sentença.

In [ ]:
#COMPLETAR ABAIXO

---
## Para casa

> Os mesmos seis verbos, com perguntas novas. O código continua no enunciado: leia, entenda **por que** é assim, e só então digite.

### Exercício C1 — Quantos processos são de 2019?

Quantos processos da base são de 2019?

Digite na célula abaixo:

```
processos.query('ano == 2019')
```

**Esperado:** `354 rows × 11 columns`.

In [ ]:
#COMPLETAR ABAIXO

### Exercício C2 — Quantos processos de pessoa jurídica são de 2023 para cá?

Processos em que o autor é pessoa jurídica **e** a sentença é de 2023 para cá.

Digite na célula abaixo:

```
processos.query('autor_tipo == "PESSOA JURÍDICA" and ano >= 2023')
```

**Esperado:** `268 rows × 11 columns`.

In [ ]:
#COMPLETAR ABAIXO

### Exercício C3 — Quem são os réus nas ações contra bancos?

Dos processos contra bancos e financeiras, só o número do processo, o réu e o setor.

Digite na célula abaixo:

```
processos.query('setor == "Banco e financeira"')[["processo", "reu", "setor"]]
```

**Esperado:** `1361 rows × 3 columns`.

In [ ]:
#COMPLETAR ABAIXO

### Exercício C4 — Em quantas sentenças houve condenação em dinheiro?

Crie a coluna `com_condenacao`, verdadeira quando `valor_condenacao` **não** está vazia, e conte os verdadeiros.

`notna()` é o contrário de `isna()`.

Digite na célula abaixo:

```
processos.assign(com_condenacao = processos["valor_condenacao"].notna())["com_condenacao"].sum()
```

**Esperado:** `1221` — só 17% das sentenças mandam pagar algum valor.

In [ ]:
#COMPLETAR ABAIXO

### Exercício C5 — Quais são as primeiras comarcas em ordem alfabética?

Ordene pela comarca e mostre as três primeiras, com o tribunal ao lado.

Digite na célula abaixo:

```
processos.sort_values("comarca")[["comarca", "tribunal"]].head(3)
```

**Esperado:** três comarcas começando por A, em ordem alfabética.

In [ ]:
#COMPLETAR ABAIXO

### Exercício C6 — Qual é a mediana da condenação em cada setor?

Média e mediana contam histórias diferentes quando há valores muito altos. Qual a **mediana** da condenação em cada setor?

Digite na célula abaixo:

```
processos.groupby("setor")["valor_condenacao"].median().sort_values(ascending=False).round(2).head(4)
```

**Esperado:** seguros 7000, poder público 5000, ensino 4000, transporte aéreo 3250.

In [ ]:
#COMPLETAR ABAIXO

### Exercício C7 — O autor ganha mais quando é pessoa física ou quando é empresa?

O autor ganha mais quando é pessoa física ou quando é empresa?

Digite na célula abaixo:

```
processos.groupby("autor_tipo")["ganhou_autor"].mean().round(3)
```

**Esperado:** pessoa física 0.314 e pessoa jurídica 0.289 — perto uma da outra, o que já é uma resposta.

In [ ]:
#COMPLETAR ABAIXO

### Exercício C8 — Como se distribuem os resultados em cada tipo de autor?

Quantas sentenças de cada resultado há em cada tipo de autor? O `groupby` aceita uma lista.

Digite na célula abaixo:

```
processos.groupby(["autor_tipo", "resultado"]).size()
```

**Esperado:** dez linhas (2 tipos de autor × 5 resultados). Em pessoa física, acordo homologado aparece 2064 vezes.

In [ ]:
#COMPLETAR ABAIXO

### Exercício C9 — O que dizem as sentenças que homologaram acordo?

Junte as tabelas de novo e mostre o dispositivo de duas sentenças que terminaram em acordo homologado.

Digite na célula abaixo:

```
completa = processos.merge(dispositivos[["processo", "dispositivo"]], on="processo", how="left")
completa.query('resultado == "ACORDO HOMOLOGADO"')[["processo", "dispositivo"]].head(2)
```

**Esperado:** duas linhas, com o número do processo e o trecho da sentença que homologou o acordo.

In [ ]:
#COMPLETAR ABAIXO

### Exercício C10 — O que decidiram as ações procedentes contra empresas aéreas?

Das ações contra empresas de transporte aéreo julgadas procedentes, quais foram os dispositivos?

Aqui entram quatro dos seis verbos, em fila.

Digite na célula abaixo:

```
completa = processos.merge(dispositivos[["processo", "dispositivo"]], on="processo", how="left")
(completa
 .query('setor == "Transporte aéreo" and resultado == "PROCEDENTE"')
 [["processo", "reu", "dispositivo"]]
 .head(3))
```

**Esperado:** três linhas: o número do processo, a empresa aérea e o trecho final da sentença.

In [ ]:
#COMPLETAR ABAIXO

---
## Fechamento

Cinco verbos, e um sexto para juntar tabelas:

| Verbo | O que faz | Como se escreve |
|---|---|---|
| **filter** | escolhe linhas | `processos.query('ano == 2022')` |
| **select** | escolhe colunas | `processos[["tribunal", "ano"]]` |
| **mutate** | cria coluna | `processos.assign(nova = ...)` |
| **arrange** | ordena | `processos.sort_values("ano")` |
| **summarise** | resume em um número | `processos.groupby("setor")["x"].mean()` |
| *join* | junta duas tabelas | `processos.merge(dispositivos, on="processo")` |

Qualquer pergunta que você fizer a uma base é uma combinação desses seis. O que muda de uma
pesquisa para outra é a **pergunta**, não o código.